# 🎵 Music Analytics Pipeline

---

## Project Objective

This project leverages the **Last.fm API** to extract real-time data on top artists and tracks globally. Using this data, we build custom performance metrics — including a **Popularity Score**, an **Underrated Score**, and a composite **Artist Performance Index** — to go beyond raw play counts and uncover deeper engagement insights.

The pipeline is designed to be **re-runnable**: each execution appends a dated snapshot to historical CSV files, enabling future **time-series analysis** and **trend tracking** across multiple runs.

---

## Technologies Used

| Tool | Purpose |
|------|---------|
| **Python 3** | Core scripting and data processing |
| **Pandas** | DataFrame manipulation, aggregation, and feature engineering |
| **Requests** | HTTP calls to the Last.fm REST API |
| **Matplotlib** | Data visualizations |
| **Last.fm API** | Source of artist and track popularity data |

---

## Imports

All required libraries are loaded at the top to keep dependencies clear and the notebook executable from start to finish.

In [ ]:
# Standard and third-party library imports
import requests
import pandas as pd
import matplotlib.pyplot as plt
import os
from datetime import date

---

## 1. Data Extraction — Top Artists

The Last.fm `chart.gettopartists` endpoint returns the most-listened-to artists globally for the current period. We extract key fields — name, play count, listener count, and profile URL — and load them into a structured DataFrame for analysis.

Understanding **who** the top artists are provides the foundation for all downstream engagement and performance calculations.

In [ ]:
# Last.fm API credentials and endpoint for top artists
api_key = "8268538dcc2c6f4c336b0cda4adc7fac"
url = f"https://ws.audioscrobbler.com/2.0/?method=chart.gettopartists&api_key={api_key}&format=json"

# Make the API request
response = requests.get(url)
data = response.json()

print(data.keys())

In [ ]:
# Extract the list of artists from the response payload
artists = data['artists']['artist']

# Confirm the number of artists returned
len(artists)

In [ ]:
# Parse artist records into a flat list of dictionaries
artist_list = []

for artist in artists:
    artist_list.append({
        'Artist': artist['name'],
        'Playcount': artist['playcount'],
        'Listeners': artist['listeners'],
        'URL': artist['url']
    })

# Build the artists DataFrame
df = pd.DataFrame(artist_list)

df.head()

In [ ]:
# Convert Playcount and Listeners from string to numeric types
df['Playcount'] = pd.to_numeric(df['Playcount'])
df['Listeners'] = pd.to_numeric(df['Listeners'])

df.info()

In [ ]:
# Preview the top 10 artists ranked by listener count
df.sort_values('Listeners', ascending=False).head(10)

---

## 2. Data Extraction — Top Tracks

The Last.fm `chart.gettoptracks` endpoint returns the most-streamed tracks globally. In addition to play and listener counts, we capture **track duration** (in seconds), which enables duration-based feature engineering and correlation analysis later in the pipeline.

In [ ]:
# Endpoint for top tracks
url = f"https://ws.audioscrobbler.com/2.0/?method=chart.gettoptracks&api_key={api_key}&format=json"

# Make the API request
response = requests.get(url)
track_data = response.json()

# Extract the list of tracks
tracks = track_data['tracks']['track']

len(tracks)

In [ ]:
# Parse track records into a flat list of dictionaries
track_list = []

for track in tracks:
    track_list.append({
        'Track': track['name'],
        'Artist': track['artist']['name'],
        'Playcount': track['playcount'],
        'Listeners': track['listeners'],
        'Duration': track['duration']
    })

# Build the tracks DataFrame
tracks_df = pd.DataFrame(track_list)

tracks_df.head()

In [ ]:
# Convert numeric columns from string to appropriate types
tracks_df['Playcount'] = pd.to_numeric(tracks_df['Playcount'])
tracks_df['Listeners'] = pd.to_numeric(tracks_df['Listeners'])
tracks_df['Duration'] = pd.to_numeric(tracks_df['Duration'])

tracks_df.info()

---

## 3. Data Storage

Saving extracted data to CSV provides a **portable, versioned snapshot** of the current API response. These files serve as the input layer for downstream tools (dashboards, SQL loaders, BI platforms) and act as a reproducible baseline for analysis without requiring repeated API calls.

In [ ]:
# Persist the top artists dataset to a CSV file
df.to_csv('top_artists.csv', index=False)

# Persist the top tracks dataset to a CSV file
tracks_df.to_csv('top_tracks.csv', index=False)

print("top_artists.csv saved successfully")
print("top_tracks.csv saved successfully")

---

## 4. Exploratory Data Analysis

Before engineering features, we explore the raw data to understand distributions, identify dominant artists, and check for any relationships between variables such as duration and listener count.

### 4.1 Artist Track Frequency

Which artists appear most frequently in the top tracks chart? A high frequency suggests sustained chart presence across multiple tracks.

In [ ]:
# Count how many top tracks each artist appears in
tracks_df['Artist'].value_counts().head(10)

### 4.2 Total Playcount by Artist

Aggregating playcounts per artist reveals cumulative streaming volume — a key measure of overall popularity.

In [ ]:
# Total playcount per artist, sorted descending
tracks_df.groupby('Artist')['Playcount'].sum().sort_values(ascending=False).head(10)

### 4.3 Combined Playcount & Listeners by Artist

This dual aggregation surfaces artists with high listener counts (broad reach) versus high playcounts (deep engagement).

In [ ]:
# Aggregate both playcount and listeners per artist
tracks_df.groupby('Artist').agg({
    'Playcount': 'sum',
    'Listeners': 'sum'
}).sort_values('Listeners', ascending=False).head(10)

### 4.4 Top Artists by Listener Count

Sorting the artists DataFrame by listeners gives us a direct chart-based ranking from the API's perspective.

In [ ]:
# Top artists ranked by total listeners
df.sort_values('Listeners', ascending=False).head(10)

### 4.5 Top Tracks by Listener Count

Similarly, ranking tracks by listeners highlights which songs have the widest audience reach.

In [ ]:
# Top tracks ranked by total listeners
tracks_df.sort_values('Listeners', ascending=False).head(10)

### 4.6 Correlation: Duration vs. Listeners

Does track length influence listener counts? A correlation coefficient near 0 suggests duration has little bearing on popularity; a strong coefficient would warrant further investigation.

In [ ]:
# Pearson correlation between track duration (seconds) and listener count
tracks_df[['Duration', 'Listeners']].corr()

---

## 5. Feature Engineering

Raw counts alone don't tell the full story. The following custom metrics reframe the data to measure **engagement quality**, **content depth**, and **relative popularity** — providing richer signals for ranking and comparison.

### 5.1 Listeners per Play

**Metric:** `Playcount / Listeners`

A higher ratio indicates that each listener replays the track more frequently — a strong signal of **deep engagement**. Tracks with many listeners but low playcounts may have broad but shallow appeal.

> Note: Intentionally computed as Playcount ÷ Listeners (plays per unique listener).

In [ ]:
# Compute plays per unique listener — measures replay depth
tracks_df['Listeners_per_Play'] = tracks_df['Playcount'] / tracks_df['Listeners']

# Tracks with the highest replay rate
tracks_df.sort_values(
    'Listeners_per_Play',
    ascending=False
).head(10)

### 5.2 Duration in Minutes

Converting track duration from seconds to minutes makes the figures human-readable and enables intuitive scatter-plot comparisons.

In [ ]:
# Convert duration from seconds to minutes
tracks_df['Minutes'] = tracks_df['Duration'] / 60

tracks_df[['Track', 'Artist', 'Minutes']].head()

### 5.3 Popularity Score

**Formula:** `(Listeners × 0.7) + (Playcount × 0.3)`

This weighted composite favours **unique reach** (listeners, 70%) over raw volume (playcount, 30%), reflecting the idea that attracting a broad audience is more indicative of mainstream popularity than heavy replay by a smaller group.

In [ ]:
# Compute weighted popularity score
tracks_df['Popularity_Score'] = (
    tracks_df['Listeners'] * 0.7 +
    tracks_df['Playcount'] * 0.3
)

# Top tracks by Popularity Score
tracks_df.sort_values(
    'Popularity_Score',
    ascending=False
)[['Track', 'Artist', 'Popularity_Score']].head(10)

### 5.4 Underrated Score

**Formula:** `(Listeners / Playcount) × 1,000,000`

Tracks with a **high listener-to-playcount ratio** attract many unique listeners relative to total plays — suggesting they are widely discovered but not yet heavily replayed. This is a proxy for tracks that may be **underrated** or on the cusp of going viral.

In [ ]:
# Compute underrated score — high listener reach relative to play volume
tracks_df['Underrated_Score'] = (
    tracks_df['Listeners'] / tracks_df['Playcount']
) * 1000000

# Top underrated tracks
tracks_df.sort_values(
    'Underrated_Score',
    ascending=False
)[['Track', 'Artist', 'Underrated_Score']].head(10)

---

## 6. Historical Data Pipeline

To enable **time-series analysis** and **trend tracking**, each pipeline run appends a dated snapshot to persistent history files. Over multiple runs, these files accumulate a longitudinal record of how artist and track rankings evolve — a foundation for forecasting and trend detection.

The `Date` column tags every row with the run date, making it straightforward to filter, group, or plot data by time period.

In [ ]:
# Stamp both DataFrames with today's run date
tracks_df['Date'] = date.today()
df['Date'] = date.today()

print("Date column added:", date.today())

### 6.1 Inspect Column Names

Confirm that the `Date` column and all engineered features are present before saving.

In [ ]:
# Verify final column set for tracks
print("Tracks columns:", tracks_df.columns.tolist())

# Verify final column set for artists
print("Artists columns:", df.columns.tolist())

### 6.2 Save Artists History

If a history file exists, the new data is appended; otherwise, a fresh file is created. This makes the pipeline **idempotent and accumulative** across runs.

In [ ]:
# Path for cumulative artists history
artist_history_path = '/Users/sanskrutivadakattu/Desktop/music-analytics-pipeline/data/artists_history.csv'

if os.path.exists(artist_history_path):
    # Append to existing history
    old_df = pd.read_csv(artist_history_path)
    combined_df = pd.concat([old_df, df], ignore_index=True)
else:
    # First run — create the history file
    combined_df = df.copy()

combined_df.to_csv(artist_history_path, index=False)

print(f"Artists history shape: {combined_df.shape}")

### 6.3 Save Tracks History

The same append-or-create logic is applied to the tracks dataset.

In [ ]:
# Path for cumulative tracks history
tracks_history_path = '/Users/sanskrutivadakattu/Desktop/music-analytics-pipeline/data/tracks_history.csv'

if os.path.exists(tracks_history_path):
    # Append to existing history
    old_tracks = pd.read_csv(tracks_history_path)
    combined_tracks = pd.concat([old_tracks, tracks_df], ignore_index=True)
else:
    # First run — create the history file
    combined_tracks = tracks_df.copy()

combined_tracks.to_csv(tracks_history_path, index=False)

print(f"Tracks history shape: {combined_tracks.shape}")

---

## 7. Statistical Summary

Descriptive statistics give a high-level view of the data's distribution — including central tendency, spread, and potential outliers. Extreme differences between mean and max values often reveal the long-tail nature of music popularity, where a handful of artists dominate the charts.

In [ ]:
# Full statistical summary for all numeric track features
tracks_df.describe()

---

## 8. Artist Performance Index

The **Artist Performance Index (API)** is a composite metric that ranks artists by combining three dimensions of performance:

| Component | Weight | What It Measures |
|-----------|--------|-----------------|
| Listeners (normalised) | 40% | Breadth of audience reach |
| Playcount (normalised) | 40% | Total streaming volume |
| Track Count (normalised) | 20% | Chart presence across multiple tracks |

### Why Normalisation?
Raw values for listeners and playcounts differ by orders of magnitude. Without normalisation, whichever variable has the largest absolute values would dominate the index regardless of its intended weight. **Min-max normalisation** scales each dimension to `[0, 1]`, ensuring the weights are applied as intended.

### 8.1 Aggregate by Artist

First, we group tracks by artist and sum listeners, playcounts, and track counts to build an artist-level summary table.

In [ ]:
# Aggregate track-level metrics up to artist level
artist_index = tracks_df.groupby('Artist').agg({
    'Listeners': 'sum',
    'Playcount': 'sum',
    'Track': 'count'
}).reset_index()

artist_index.head()

### 8.2 Normalise and Compute Artist Performance Index

Each component is scaled to `[0, 1]` before applying the weighted formula.

In [ ]:
# Normalise each metric to [0, 1] range
artist_index['Listeners_Norm'] = (
    artist_index['Listeners'] /
    artist_index['Listeners'].max()
)

artist_index['Playcount_Norm'] = (
    artist_index['Playcount'] /
    artist_index['Playcount'].max()
)

artist_index['Track_Norm'] = (
    artist_index['Track'] /
    artist_index['Track'].max()
)

# Apply weighted composite formula: 40% listeners + 40% playcount + 20% track presence
artist_index['Artist_Performance_Index'] = (
    0.4 * artist_index['Listeners_Norm'] +
    0.4 * artist_index['Playcount_Norm'] +
    0.2 * artist_index['Track_Norm']
)

# Display top 10 artists by Artist Performance Index
artist_index.sort_values(
    'Artist_Performance_Index',
    ascending=False
)[['Artist', 'Artist_Performance_Index']].head(10)

---

## 9. Visualizations

Visual exploration helps validate and communicate the patterns uncovered during analysis. The charts below cover artist-level performance rankings, the relationship between playcounts and listener counts, and the effect of track duration on audience size.

### 9.1 Top Artists by Artist Performance Index

This bar chart ranks artists using the composite API score, providing an at-a-glance comparison of overall chart dominance.

In [ ]:
# Bar chart: Top 10 artists ranked by Artist Performance Index
ax = artist_index.sort_values(
    'Artist_Performance_Index',
    ascending=False
).head(10).plot(
    x='Artist',
    y='Artist_Performance_Index',
    kind='bar',
    figsize=(12, 5),
    color='steelblue',
    title='Top 10 Artists by Artist Performance Index'
)

ax.set_xlabel('Artist')
ax.set_ylabel('Artist Performance Index')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('artist_api.png', dpi=150)
plt.show()

**Interpretation:** Artists with the highest API scores demonstrate strong performance across all three dimensions — broad audience, high stream volume, and consistent chart presence. A large gap between the top artist and the rest indicates a dominant chart leader.

### 9.2 Playcount vs. Listeners

This scatter plot explores whether tracks with more listeners also accumulate proportionally more plays, or whether some tracks have disproportionately high play-to-listener ratios.

In [ ]:
# Scatter plot: Playcount vs. Listeners
ax = tracks_df.plot.scatter(
    x='Playcount',
    y='Listeners',
    figsize=(9, 5),
    alpha=0.6,
    color='tomato',
    title='Playcount vs. Listeners'
)

ax.set_xlabel('Playcount')
ax.set_ylabel('Listeners')
plt.tight_layout()
plt.show()

**Interpretation:** A roughly linear cluster suggests most tracks scale naturally between plays and listeners. Outliers above the trend line represent high-replay tracks (deep engagement); outliers below it indicate widely discovered but rarely replayed tracks.

### 9.3 Track Duration vs. Listeners

Does track length affect how many people listen? This chart tests whether shorter or longer tracks tend to attract larger audiences.

In [ ]:
# Scatter plot: Duration (minutes) vs. Listeners
ax = tracks_df.plot.scatter(
    x='Minutes',
    y='Listeners',
    figsize=(9, 5),
    alpha=0.6,
    color='mediumseagreen',
    title='Track Duration (Minutes) vs. Listeners'
)

ax.set_xlabel('Duration (Minutes)')
ax.set_ylabel('Listeners')
plt.tight_layout()
plt.show()

**Interpretation:** If listener counts are evenly distributed across duration values, duration has little influence on popularity — consistent with the near-zero correlation observed in Section 4.6. Any visible cluster in the 3–4 minute range reflects the industry standard for commercially successful track lengths.

---

## 10. Key Findings

Based on the analysis performed across this pipeline, the following insights emerge:

1. **Listener concentration is highly skewed.** A small number of artists account for a disproportionate share of total listeners and playcounts, exhibiting a classic long-tail distribution in music consumption.

2. **Replay depth varies significantly.** The `Listeners_per_Play` metric reveals that some tracks attract wide audiences who listen infrequently, while others build smaller but highly loyal, high-replay fan bases — a meaningful distinction for artist strategy.

3. **Duration does not predict popularity.** The near-zero correlation between track duration and listener count suggests that song length is not a driver of chart performance in this dataset.

4. **Multi-track chart presence amplifies the Artist Performance Index.** Artists appearing across several top tracks benefit from the Track Norm component of the API, rewarding consistent output over one-hit performance.

5. **Underrated Score surfaces emerging tracks.** Tracks ranking highly on `Underrated_Score` have broad reach relative to their replay volume — a potential early-detection signal for rising popularity before full mainstream breakout.

---

## 11. Future Enhancements

This pipeline is designed to evolve. The following enhancements are planned for subsequent development phases:

| Enhancement | Description |
|-------------|-------------|
| **Automated Daily Extraction** | Schedule the notebook via Airflow or a cron job to run daily, continuously populating the history files without manual execution. |
| **Interactive Dashboard** | Build a Streamlit or Power BI dashboard on top of the CSV exports, enabling real-time filtering by artist, date range, and metric. |
| **Snowflake Integration** | Replace local CSV storage with a Snowflake data warehouse to support scalable, SQL-queryable historical data and team collaboration. |
| **Trend Analysis** | Use the accumulated `artists_history.csv` and `tracks_history.csv` files to plot ranking trajectories over time and identify rising or declining artists. |
| **Forecasting** | Apply time-series models (e.g., Prophet, ARIMA) to listener and playcount trends, enabling short-term popularity forecasts for newly charting artists and tracks. |

---
*Pipeline authored as part of a Data Analytics portfolio project.*